#Predict Flight Delays at Albuquerque International Sunport (ABQ)

##Problem Definition

This is a **Supervised Regression** machine learning problem. The objective is to develop models that **predict the arrival delay rate at Albuquerque International Sunport (ABQ)**. The delay rate is defined as the proportion of arriving flights that are delayed by 15 minutes or more during a given month for a specific airline. The models will use airline characteristics, seasonal trends, weather conditions, and airport operational data to identify the factors that contribute to flight delays and estimate future delay rates.

Several regression models will be evaluated, including **Linear Regression**, **Random Forest Regression**, and **XGBoost Regression**. Linear Regression will serve as the baseline model, while Random Forest and XGBoost will capture more complex, nonlinear relationships in the data. XGBoost incorporates regularization to help reduce overfitting, and both tree-based models provide feature importance scores that can be used to identify the variables that contribute most to prediction performance.

Model performance will be evaluated using cross-validated **Root Mean Squared Error (RMSE)** as the primary metric. Additional evaluation metrics, including **Mean Absolute Error (MAE)** and the **coefficient of determination (R²)**, will also be reported to compare model performance and predictive **accuracy**.

##Data Collection

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from google.colab import userdata
import os

In [2]:
hf_url = 'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/df.parquet'
hf_url

'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/df.parquet'

In [3]:
df = pd.read_parquet(hf_url)
# df.head().sort_values(by=['year', 'month'])
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
year,884.0,2022.595023,1.853349,2020.0,2021.00,2022.00,2024.000,2026.00
month,884.0,6.223982,3.470577,1.0,3.00,6.00,9.000,12.00
arr_flights,883.0,159.141563,230.224226,1.0,30.00,62.00,190.000,1186.00
arr_del15,882.0,31.082766,52.393397,0.0,4.00,12.00,32.750,330.00
carrier_ct,883.0,12.427826,19.552900,0.0,1.01,4.33,14.300,117.53
weather_ct,883.0,0.976863,2.242832,0.0,0.00,0.00,1.000,25.00
nas_ct,883.0,4.751087,6.559453,0.0,0.71,2.53,6.195,51.58
security_ct,883.0,0.060668,0.259608,0.0,0.00,0.00,0.000,2.64
late_aircraft_ct,883.0,12.831200,27.799570,0.0,0.77,3.00,9.415,195.25
arr_cancelled,883.0,2.793884,14.982775,0.0,0.00,0.00,2.000,384.00


In [4]:
df.shape

(884, 21)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 884 entries, 0 to 883
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 884 non-null    int64  
 1   month                884 non-null    int64  
 2   carrier              884 non-null    object 
 3   carrier_name         884 non-null    object 
 4   airport              884 non-null    object 
 5   airport_name         884 non-null    object 
 6   arr_flights          883 non-null    float64
 7   arr_del15            882 non-null    float64
 8   carrier_ct           883 non-null    float64
 9   weather_ct           883 non-null    float64
 10  nas_ct               883 non-null    float64
 11  security_ct          883 non-null    float64
 12  late_aircraft_ct     883 non-null    float64
 13  arr_cancelled        883 non-null    float64
 14  arr_diverted         883 non-null    float64
 15  arr_delay            883 non-null    flo

##Data Cleaning

###Rows

In [6]:
#Rows w/Nulls
df.isnull().sum().sort_values()

,0
year,0
month,0
carrier,0
carrier_name,0
airport,0
airport_name,0
arr_flights,1
carrier_ct,1
late_aircraft_ct,1
weather_ct,1


In [7]:
#Missing Rows
df[df.isnull().any(axis=1)]

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
647,2021,8,UA,United Air Lines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
834,2020,4,G4,Allegiant Air,ABQ,"Albuquerque, NM: Albuquerque International Sun...",4.0,NaN,0.0,0.0,...,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
df.drop(647, inplace=True)

In [9]:
df.loc[834, df.loc[834].isnull()]

,834
arr_del15,NaN


In [10]:
df['arr_del15'] = df['arr_del15'].fillna(0.0)

In [11]:
df[df.isnull().any(axis=1)]

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay


In [12]:
#Duplicate rows
df.duplicated().sum()

np.int64(0)

####Target

In [13]:
print(df['arr_del15'].isna().sum())
print(df['arr_flights'].isna().sum())

0
0


In [14]:
df['delay_rate'] = df['arr_del15'] / df['arr_flights']

In [15]:
target = 'delay_rate'

In [16]:
df[target].isna().sum()

np.int64(0)

In [17]:
df[target]

,delay_rate
0,0.232558
1,0.146809
2,0.032258
3,0.228448
4,0.316667
...,...
879,0.129825
880,0.162162
881,0.072645
882,0.147651


In [18]:
df[target].describe().transpose()

,delay_rate
count,883.000000
mean,0.188967
std,0.134383
min,0.000000
25%,0.098163
50%,0.170455
75%,0.250000
max,1.000000


####Unique IDs

In [19]:
#Run code to identifier columns
identifier_cols = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_cols.append(col)

print(identifier_cols)

[]


###Columns

In [20]:
#Other possible targets
delays_to_conver = [
  'arr_del15',
  'arr_flights',
  'carrier_delay',
  'weather_delay',
  'nas_delay',
  'security_delay',
  'late_aircraft_delay'
]

In [21]:
df.dtypes.value_counts()

,count
float64,16
object,4
int64,2


####Categorical

In [22]:
df_obj = df.select_dtypes(include=['object'])
df_obj

,carrier,carrier_name,airport,airport_name
0,MQ,Envoy Air,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
1,OO,SkyWest Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
2,QX,Horizon Air,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
3,UA,United Air Lines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
4,AA,American Airlines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
...,...,...,...,...
879,OO,SkyWest Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
880,UA,United Air Lines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
881,WN,Southwest Airlines,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
882,YV,Mesa Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun..."


In [23]:
#Dictionary for carrier name before drop
carrier_dict = dict(zip(df['carrier'], df['carrier_name']))
dict(sorted(carrier_dict.items()))

{'AA': 'American Airlines Network',
 'AS': 'Alaska Airlines Network',
 'AX': 'Trans States Airlines',
 'B6': 'JetBlue Airways',
 'C5': 'Commutair Aka Champlain Enterprises, Inc.',
 'CP': 'Compass Airlines',
 'DL': 'Delta Air Lines Network',
 'EV': 'ExpressJet Airlines LLC',
 'F9': 'Frontier Airlines',
 'G4': 'Allegiant Air',
 'G7': 'GoJet Airlines LLC d/b/a United Express',
 'MQ': 'Envoy Air',
 'NK': 'Spirit Airlines',
 'OO': 'SkyWest Airlines Inc.',
 'QX': 'Horizon Air',
 'UA': 'United Air Lines Network',
 'WN': 'Southwest Airlines',
 'YV': 'Mesa Airlines Inc.',
 'YX': 'Republic Airline'}

In [24]:
df.drop(columns=[
  'carrier_name',
  'airport_name',
  'airport'
], inplace=True)

In [25]:
df_obj = df.select_dtypes(include=['object'])
df_obj.sort_values(by='carrier')

,carrier
437,AA
156,AA
153,AA
737,AA
138,AA
...,...
608,YX
851,YX
29,YX
676,YX


In [26]:
df_obj.isna().sum()

,0
carrier,0


####DateTime

In [27]:
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df['date'].head()

,date
0,2026-05-01
1,2026-05-01
2,2026-05-01
3,2026-05-01
4,2026-05-01


In [28]:
df['years'] = df['date'].dt.year
df['years'].head(2)

,years
0,2026
1,2026


In [29]:
df['months'] = df['date'].dt.month
df['months'].head(2)

,months
0,5
1,5


In [30]:
df.drop(columns=['year','month'], inplace=True)

In [31]:
df.head()

,carrier,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_rate,date,years,months
0,MQ,43.0,10.0,4.00,0.94,3.19,0.0,1.87,0.0,1.0,701.0,290.0,116.0,118.0,0.0,177.0,0.232558,2026-05-01,2026,5
1,OO,470.0,69.0,43.05,6.10,11.05,0.0,8.80,1.0,2.0,4027.0,2525.0,360.0,498.0,0.0,644.0,0.146809,2026-05-01,2026,5
2,QX,31.0,1.0,1.00,0.00,0.00,0.0,0.00,0.0,0.0,46.0,46.0,0.0,0.0,0.0,0.0,0.032258,2026-05-01,2026,5
3,UA,232.0,53.0,21.40,0.89,9.24,0.0,21.47,2.0,2.0,2784.0,996.0,47.0,365.0,0.0,1376.0,0.228448,2026-05-01,2026,5
4,AA,240.0,76.0,28.22,3.45,9.49,0.0,34.83,9.0,0.0,5019.0,1663.0,452.0,437.0,0.0,2467.0,0.316667,2026-05-01,2026,5


####Numerical

#####Floats

In [32]:
df_flt = df.select_dtypes(include=['float64']).drop(columns=[target])
df_flt.head()

,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,43.0,10.0,4.00,0.94,3.19,0.0,1.87,0.0,1.0,701.0,290.0,116.0,118.0,0.0,177.0
1,470.0,69.0,43.05,6.10,11.05,0.0,8.80,1.0,2.0,4027.0,2525.0,360.0,498.0,0.0,644.0
2,31.0,1.0,1.00,0.00,0.00,0.0,0.00,0.0,0.0,46.0,46.0,0.0,0.0,0.0,0.0
3,232.0,53.0,21.40,0.89,9.24,0.0,21.47,2.0,2.0,2784.0,996.0,47.0,365.0,0.0,1376.0
4,240.0,76.0,28.22,3.45,9.49,0.0,34.83,9.0,0.0,5019.0,1663.0,452.0,437.0,0.0,2467.0


In [33]:
flt_cols = df_flt.columns.to_list()
flt_cols

['arr_flights',
 'arr_del15',
 'carrier_ct',
 'weather_ct',
 'nas_ct',
 'security_ct',
 'late_aircraft_ct',
 'arr_cancelled',
 'arr_diverted',
 'arr_delay',
 'carrier_delay',
 'weather_delay',
 'nas_delay',
 'security_delay',
 'late_aircraft_delay']

In [34]:
flt_to_int = []

for col in flt_cols:
    if (df_flt[col] % 1 != 0).any() == False:
        flt_to_int.append(col)

print(flt_to_int)

['arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']


In [35]:
df[flt_to_int] = df[flt_to_int].astype('int16')
df[flt_to_int]

,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,43,10,0,1,701,290,116,118,0,177
1,470,69,1,2,4027,2525,360,498,0,644
2,31,1,0,0,46,46,0,0,0,0
3,232,53,2,2,2784,996,47,365,0,1376
4,240,76,9,0,5019,1663,452,437,0,2467
...,...,...,...,...,...,...,...,...,...,...
879,285,37,2,0,1791,902,89,425,0,375
880,74,12,0,0,586,55,0,218,0,313
881,881,64,6,2,2838,1346,0,326,0,1166
882,149,22,1,0,1837,228,131,139,0,1339


In [36]:
flt_to_flt = df.select_dtypes(include=['float']).drop(columns=[target])
flt_to_flt

,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct
0,4.00,0.94,3.19,0.0,1.87
1,43.05,6.10,11.05,0.0,8.80
2,1.00,0.00,0.00,0.0,0.00
3,21.40,0.89,9.24,0.0,21.47
4,28.22,3.45,9.49,0.0,34.83
...,...,...,...,...,...
879,23.36,1.89,8.74,0.0,3.00
880,1.96,0.00,3.44,0.0,6.60
881,28.64,0.00,9.80,0.0,25.56
882,7.53,0.90,2.95,0.0,10.61


In [37]:
df[['security_ct', 'weather_ct', 'nas_ct']] = df[['security_ct', 'weather_ct', 'nas_ct']].astype('float32').round(2)
df[['security_ct', 'weather_ct', 'nas_ct']].head()

,security_ct,weather_ct,nas_ct
0,0.0,0.94,3.19
1,0.0,6.10,11.05
2,0.0,0.00,0.00
3,0.0,0.89,9.24
4,0.0,3.45,9.49


In [38]:
df_flt = df.select_dtypes(include=['float']).drop(columns=[target])
df_flt.head()

,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct
0,4.00,0.94,3.19,0.0,1.87
1,43.05,6.10,11.05,0.0,8.80
2,1.00,0.00,0.00,0.0,0.00
3,21.40,0.89,9.24,0.0,21.47
4,28.22,3.45,9.49,0.0,34.83


In [39]:
df.dtypes

,0
carrier,object
arr_flights,int16
arr_del15,int16
carrier_ct,float64
weather_ct,float32
nas_ct,float32
security_ct,float32
late_aircraft_ct,float64
arr_cancelled,int16
arr_diverted,int16


#####Integers

In [40]:
df_int = df.select_dtypes(include=['integer'])
df_int.head()

,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,years,months
0,43,10,0,1,701,290,116,118,0,177,2026,5
1,470,69,1,2,4027,2525,360,498,0,644,2026,5
2,31,1,0,0,46,46,0,0,0,0,2026,5
3,232,53,2,2,2784,996,47,365,0,1376,2026,5
4,240,76,9,0,5019,1663,452,437,0,2467,2026,5


In [41]:
df[['arr_diverted']].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
arr_diverted,883.0,0.198188,0.555652,0.0,0.0,0.0,0.0,5.0


In [42]:
df[['arr_diverted']] = df[['arr_diverted']].astype('int8')
df[['arr_diverted']]

,arr_diverted
0,1
1,2
2,0
3,2
4,0
...,...
879,0
880,0
881,2
882,0


In [43]:
df_int = df.select_dtypes(include=['integer'])
df_int

,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,years,months
0,43,10,0,1,701,290,116,118,0,177,2026,5
1,470,69,1,2,4027,2525,360,498,0,644,2026,5
2,31,1,0,0,46,46,0,0,0,0,2026,5
3,232,53,2,2,2784,996,47,365,0,1376,2026,5
4,240,76,9,0,5019,1663,452,437,0,2467,2026,5
...,...,...,...,...,...,...,...,...,...,...,...,...
879,285,37,2,0,1791,902,89,425,0,375,2020,1
880,74,12,0,0,586,55,0,218,0,313,2020,1
881,881,64,6,2,2838,1346,0,326,0,1166,2020,1
882,149,22,1,0,1837,228,131,139,0,1339,2020,1


In [44]:
df_int.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
arr_flights,883.0,159.141563,230.224226,1.0,30.0,62.0,190.0,1186.0
arr_del15,883.0,31.047565,52.374134,0.0,4.0,12.0,32.5,330.0
arr_cancelled,883.0,2.793884,14.982775,0.0,0.0,0.0,2.0,384.0
arr_diverted,883.0,0.198188,0.555652,0.0,0.0,0.0,0.0,5.0
arr_delay,883.0,1761.135900,2867.165553,0.0,160.5,612.0,1828.0,19855.0
carrier_delay,883.0,683.990940,1038.917589,0.0,51.0,220.0,841.0,5996.0
weather_delay,883.0,78.949037,179.813795,0.0,0.0,0.0,69.0,2248.0
nas_delay,883.0,188.272933,267.014219,0.0,20.0,87.0,243.0,1907.0
security_delay,883.0,2.390713,12.073293,0.0,0.0,0.0,0.0,192.0
late_aircraft_delay,883.0,807.532276,1630.665674,0.0,22.0,183.0,652.5,11744.0


In [45]:
df.dtypes.value_counts()

,count
int16,9
float64,3
float32,3
int32,2
object,1
int8,1
datetime64[ns],1


###Features

In [46]:
features = df[['years', 'months', 'carrier']]

###One-Hot Encoding

In [47]:
categorical_features = ['carrier']

preprocessor = ColumnTransformer(
  transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
  ], remainder='passthrough'
)

In [48]:
features.shape

(883, 3)

In [49]:
features_encoded = preprocessor.fit_transform(features)

In [50]:
features_encoded.shape

(883, 21)

In [51]:
preprocessor.get_feature_names_out()

array(['cat__carrier_AA', 'cat__carrier_AS', 'cat__carrier_AX',
       'cat__carrier_B6', 'cat__carrier_C5', 'cat__carrier_CP',
       'cat__carrier_DL', 'cat__carrier_EV', 'cat__carrier_F9',
       'cat__carrier_G4', 'cat__carrier_G7', 'cat__carrier_MQ',
       'cat__carrier_NK', 'cat__carrier_OO', 'cat__carrier_QX',
       'cat__carrier_UA', 'cat__carrier_WN', 'cat__carrier_YV',
       'cat__carrier_YX', 'remainder__years', 'remainder__months'],
      dtype=object)

In [52]:
features_encoded_df = pd.DataFrame(
    features_encoded.toarray(),
    columns=preprocessor.get_feature_names_out()
)

features_encoded_df.head()

,cat__carrier_AA,cat__carrier_AS,cat__carrier_AX,cat__carrier_B6,cat__carrier_C5,cat__carrier_CP,cat__carrier_DL,cat__carrier_EV,cat__carrier_F9,cat__carrier_G4,...,cat__carrier_MQ,cat__carrier_NK,cat__carrier_OO,cat__carrier_QX,cat__carrier_UA,cat__carrier_WN,cat__carrier_YV,cat__carrier_YX,remainder__years,remainder__months
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026.0,5.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2026.0,5.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0


In [53]:
[col for col in features_encoded_df.columns if 'carrier' in col]

['cat__carrier_AA',
 'cat__carrier_AS',
 'cat__carrier_AX',
 'cat__carrier_B6',
 'cat__carrier_C5',
 'cat__carrier_CP',
 'cat__carrier_DL',
 'cat__carrier_EV',
 'cat__carrier_F9',
 'cat__carrier_G4',
 'cat__carrier_G7',
 'cat__carrier_MQ',
 'cat__carrier_NK',
 'cat__carrier_OO',
 'cat__carrier_QX',
 'cat__carrier_UA',
 'cat__carrier_WN',
 'cat__carrier_YV',
 'cat__carrier_YX']

###Make a copy

In [54]:
df_clean = features_encoded_df.copy()

In [55]:
df_clean = df_clean.rename(columns={
  'remainder__years':'year',
  'remainder__months':'month'
})

In [56]:
df_clean['delay_rate'] = df[target].values

In [57]:
df_clean.head()

,cat__carrier_AA,cat__carrier_AS,cat__carrier_AX,cat__carrier_B6,cat__carrier_C5,cat__carrier_CP,cat__carrier_DL,cat__carrier_EV,cat__carrier_F9,cat__carrier_G4,...,cat__carrier_NK,cat__carrier_OO,cat__carrier_QX,cat__carrier_UA,cat__carrier_WN,cat__carrier_YV,cat__carrier_YX,year,month,delay_rate
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0,0.232558
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0,0.146809
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2026.0,5.0,0.032258
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2026.0,5.0,0.228448
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026.0,5.0,0.316667


#Save as a Parquet

In [58]:
parquet_file = 'BTS.parquet'
parquet_file

'BTS.parquet'

In [59]:
df_clean.to_parquet(parquet_file, index=False)

In [60]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 22k Jul 31 01:57 BTS.parquet


In [61]:
df2 = pd.read_parquet(parquet_file)
df2.shape

(883, 22)

In [62]:
os.environ["HF_TOKEN"] = userdata.get('hf_cs_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_ld ... iLe'

In [63]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account

'stephanie465337'

In [64]:
hf_org = "ddds-Capstone"
os.environ["HF_ORG"] = hf_org
hf_org

'ddds-Capstone'

In [65]:
hf_repo = "Datasets"
os.environ["HF_REPO"] = hf_repo
hf_repo

'Datasets'

In [66]:
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Capstone Token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [67]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ORG}/${HF_REPO} \
  BTS.parquet

In [68]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/ddds-Capstone/Datasets/commit/78c5c7823b71566837194a0d1f20104d2e550927



In [69]:
hf_url = f"https://huggingface.co/datasets/{hf_org}/{hf_repo}/resolve/main/BTS.parquet"
hf_url

'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/BTS.parquet'

In [70]:
df3 = pd.read_parquet(hf_url)
df3.shape

(883, 22)

In [71]:
df3.iloc[:,:5].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 883 entries, 0 to 882
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cat__carrier_AA  883 non-null    float64
 1   cat__carrier_AS  883 non-null    float64
 2   cat__carrier_AX  883 non-null    float64
 3   cat__carrier_B6  883 non-null    float64
 4   cat__carrier_C5  883 non-null    float64
dtypes: float64(5)
memory usage: 34.6 KB
